In [ ]:


import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

def etl_pipeline():
    print(" Starting ETL Pipeline...")
    
    # 1. EXTRACT
    print(" 1. EXTRACT: Loading Titanic dataset...")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print(f"   Raw shape: {df.shape}")
    
    # Select columns
    cols = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    df = df[cols].copy()
    
    # 2. TRANSFORM
    print(" 2. TRANSFORM: Building preprocessing pipeline...")
    
    numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
    categorical_features = ['Pclass', 'Sex', 'Embarked']
    target = 'Survived'
    
    # Pipelines
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
    
    # Transform data
    X = df.drop(target, axis=1)
    y = df[target]
    X_transformed = preprocessor.fit_transform(X)
    
    # Feature names
    cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
    all_feature_names = numeric_features + list(cat_feature_names)
    
    # Final DataFrame
    X_df = pd.DataFrame(X_transformed, columns=all_feature_names, index=df.index)
    processed_df = pd.concat([pd.Series(y.values, name=target, index=df.index), X_df], axis=1)
    
    print(f"   Processed shape: {processed_df.shape}")
    print("   Missing values: 0")
    
    # 3. LOAD (CSV only - no PyArrow needed)
    print(" 3. LOAD: Saving files...")
    processed_df.to_csv('titanic_etl_processed.csv', index=False)
    
    # Save pipeline
    try:
        import joblib
        joblib.dump(preprocessor, 'etl_preprocessor.pkl')
        print("- etl_preprocessor.pkl")
    except ImportError:
        print("   Note: Install joblib for pipeline saving: pip install joblib")
    
    print("- titanic_etl_processed.csv ✓")
    print(" ETL Complete!")
    return processed_df

# RUN
if __name__ == "__main__":
    final_df = etl_pipeline()
    print("\nFirst 5 rows of processed data:")
    print(final_df.head())
